In [4]:

import numpy as np
def analyze_matrix(A) -> dict:#khai báo
       A=np.array(A, dtype=float) #ma trận A
    #loại trường hợp ma trận không vuông
       if A.ndim != 2 or A.shape[0] != A.shape[1]:
            raise ValueError("A phải là ma trận vuông") 
    #kiểm tra trường hợp phần tử đường chéo =0
       n=A.shape[0]
       diag=np.abs(np.diag(A))
       if np.any(diag<1e-12):  #1e-12: 0,0000000000000001 
            raise ValueError("phần tử đường chéo bằng 0, không tính được q và q'")
    #tính tổng ngoài đường chéo bruh
       abs_A= np.abs(A)
       off_row=abs_A.sum(axis=1)-diag
       off_col=abs_A.sum(axis=0)-diag
    #kiểm tra chéo trội theo hàng go bruh bruh
       row_dominant=bool(np.all(diag >=off_row -1e-12))
       row_strict=bool(np.all(diag > off_row + 1e-12)) #chéo trội nghiêm ngặt
    #kiểm tra chéo trội theo cột go bruh bruh
       col_dominant=bool(np.all(diag >=off_col -1e-12))
       col_strict=bool(np.all(diag > off_col + 1e-12)) #nt
    #tính q
       q=float(np.max(off_row/diag))
    #tính q guass seidel q_gs
       mauso=diag-np.tril(abs_A,k=-1).sum(axis=1)#tril: ma trận tam giác dưới
                                               #tổng trái theo công thức tính tay
       if np.any(mauso <= 1e-12):##kiểm tra mẫu số =0
           q_gs=float("inf")##một phần tử vi phạm gán inf
       else:
           q_gs=float(np.max(np.triu(abs_A,k=1).sum(axis=1)/mauso))#triu: ma trận tam giác trên
                          #tổng phải theo công thức tính tay
    ## chuẩn Bvc
       B=np.zeros((n,n))
       for i in range(n):
           for j in range(n):
               if j!=i:
                   B[i,j]=-A[i,j]/A[i,i]
       chuanvcB=0.0
       for i in range(n):
           tong=sum(abs(B[i,j]) for j in range(n)) ### tổng trị tuyệt đối theo hàng
           chuanvcB=max(chuanvcB,tong)
    #kiểm tra đối xứng
       dx=np.allclose(A,A.T, atol=1e-12)
    #bán kính phổ
       bkpho=float(np.max(np.abs(np.linalg.eigvals(B)))) # max của giá trị 
       bkpho=round(bkpho,6)
       return {
            "row_dominant": row_dominant, 
            "row_strict": row_strict,
            "col_dominant": col_dominant, 
            "col_strict": col_strict,
            "q": q, 
            "q_gaussseidel": q_gs, 
            "chuanB": chuanvcB,
            "doixungg": dx, 
            "Bankinhpho": bkpho, ## làm tròn 6 chữ số
            "matranB": B,
        }

In [5]:
A=[[10,-1,2,0],
   [-1,11,-1,3],
   [2,-1,10,-1],
   [0,3,-1,8]]
dapan=analyze_matrix(A)
for k,v in dapan.items():
    if k!="matranB":
        print(k,":",v)

row_dominant : True
row_strict : True
col_dominant : True
col_strict : True
q : 0.5
q_gaussseidel : 0.4
chuanB : 0.5
doixungg : True
Bankinhpho : 0.426437


In [7]:

import pytest
import numpy as np



# assert.... thỏa mãn nếu điều kiện đúng, sai thì test thất bại
# ma trận B[i][j] = −A[i][j] / A[i][i] khi j ≠ i, và B[i][i] = 0


# 1 ví dụ chuẩn (tính tay được)
def test_vi_du_chuan():
    A = [[4, 1, 1],
         [1, 5, 2],
         [0, 1, 3]]
    r = analyze_matrix(A)  # hàng 1: 4>2 | hàng 2: 5>3 | hàng 3: 3>1 -> chéo trội nghiêm ngặt
                           # cột 3: 3 = 3 nên trội nhưng không nghiêm ngặt
    assert r["row_dominant"]
    assert r["row_strict"]          # hàng 1: 4>2 | hàng 2: 5>3 | hàng 3: 3>1 -> chéo trội nghiêm ngặt
    assert r["col_dominant"]
    assert not r["col_strict"]
    assert r["q"] == pytest.approx(0.6)
    assert r["q_gaussseidel"] == pytest.approx(0.5)
    assert r["chuanB"] == pytest.approx(0.6)
    assert not r["doixungg"]

    expected_B = np.array([[0, -0.25, -0.25],  # hàng 0
                           [-0.2, 0, -0.4],    # hàng 1
                           [0, -1/3, 0]])      # hàng 2
    np.testing.assert_allclose(r["matranB"], expected_B)


# 2 ma trận đường chéo

# 2  0
# 0  5
def test_ma_tran_duong_cheo():
    r = analyze_matrix([[2, 0], [0, 5]])
    assert r["q"] == 0.0               # q=0 (tổng phần tử ngoài đường chéo / đường chéo)
    assert r["q_gaussseidel"] == 0.0   # q_gs=0
    assert r["Bankinhpho"] == 0.0      # bk phổ = 0
    assert r["doixungg"]               # ma trận đường chéo đối xứng
    assert r["row_strict"]
    assert r["col_strict"]


# 3 ma trận đối xứng 2x2
def test_doi_xung_2x2():

    # A=   2  1
    #      1  2

    r = analyze_matrix([[2, 1], [1, 2]])
    # B= 0     -0,5
    #   -0,5    0
    assert r["doixungg"]
    assert r["q"] == pytest.approx(0.5)
    assert r["q_gaussseidel"] == pytest.approx(0.5)  # tổng phải / ( đường chéo - tổng trái )
    assert r["Bankinhpho"] == pytest.approx(0.5)     # giá trị riêng của B: 0,5
    # Giá trị riêng lamda tạm kí hiệu là: h
    # giải det(B-hI)=0    I là ma trận đơn vị  1 0
    #                                          0 1
    # có hI=   h      0
    #          0      h
    # B-hI =   -h      -0,5
    #          -0,5    -h
    # tính det(B-hI)= h^2 - 0,25 =0 => h= +- 0,5. Vậy giá trị riêng là ±0,5


# 4 không chéo trội
def test_khong_cheo_troi():
    r = analyze_matrix([[1, 2], [3, 1]])
    # 1   2
    # 3   1
    # B=  0   -2
    #     -3  0
    assert not r["row_dominant"]  # 1<2 đường chéo nhỏ hơn -> không trội hàng
    assert not r["col_dominant"]  # 1<3 đường chéo nhỏ hơn -> không trội cột
    assert r["q"] == pytest.approx(3.0)  # max(2/1 , 3/1)=3
    assert r["Bankinhpho"] > 1
    assert r["Bankinhpho"] == pytest.approx(2.449490)  # căn 6 > 1
    # hI= h   0
    #     0   h
    # B-hI=     -h  -2
    #           -3   -h
    # det(B-hI)= h^2-6=0  -> h=±căn6
    assert r["q_gaussseidel"] == float("inf")
    # nếu mẫu âm thì không dùng được đánh giá q_gs, gán = inf (vô cùng) để không đảm bảo hội tụ. Vì cần q_gs < 1
    # nếu mẫu bé hơn 0 thì không chia được / không đánh giá co được
    # vì sao phải chặn? Ma trận A không hề trội, bán kính phổ lớn hơn 1. Nếu cứ chia máy sẽ trả kết luận giả
    # q_gs = vô cùng thì phân kì? Không kết luận được, cần xét thêm bán kính phổ nhỏ hơn 1
    # còn nếu A chéo trội nghiêm ngặt thì chắc chắn mẫu dương


# 5 chéo trội không nghiêm ngặt (lớn hơn hoặc bằng)
def test_cheo_troi_dau_bang():
    r = analyze_matrix([[2, 2], [1, 3]])
    # A = 2   2
    #     1   3
    assert r["row_dominant"]
    assert not r["row_strict"]  # hàng 1: 2 = 2


# 6 Đường chéo âm (dùng trị tuyệt đối khi so sánh)
def test_duong_cheo_am():
    r = analyze_matrix([[-4, 1], [1, -4]])
    # A = -4   1
    #      1  -4
    # B = 0      0,25
    #     0,25     0
    # hI = h     0
    #      0     h
    # B-hI = -h   0,25
    #        0,25  -h
    # det(B-hI)= h^2 - 0,0625 -> h= ±0,25
    assert r["row_strict"]  # |-4|>1 chéo trội nghiêm ngặt
    assert r["q"] == pytest.approx(0.25)
    assert r["Bankinhpho"] == pytest.approx(0.25)
    assert r["matranB"][0, 1] == pytest.approx(0.25)
    # dấu của B phải đúng: -1 / -4 = +0.25  (hàng 0 cột 1 phải là 0,25)


# 7 Giá trị riêng phức
def test_gia_tri_rieng_phuc():
    r = analyze_matrix([[1, 2], [-2, 1]])
    # A = 1    2
    #    -2    1
    # B=  0    -2
    #     2    0
    # hI= h   0
    #     0   h
    # B-hI= -h   -2
    #        2   -h
    # det(B-hI)= h^2 + 4 -> h=2i và -2i
    # but |2i|=2. Test để chắc chắn máy dùng np.abs đúng cho số phức
    assert r["Bankinhpho"] == pytest.approx(2.0)


# 8 Kiểm tra làm tròn 6 chữ số
def test_lam_tron_6_chu_so():
    r = analyze_matrix([[4, 1, 1], [1, 5, 2], [0, 1, 3]])
    pho = r["Bankinhpho"]
    assert pho == round(pho, 6)


# 9 bkpho(B) <= q, kiểm tra 1205 trường hợp khác nhau để bkpho luôn nhỏ hơn hoặc bằng q
def test_bkpho_nho_hon_hoac_bang_q():
    np.random.seed(0)  # seed 0 là để cố định kết quả ngẫu nhiên
    for _ in range(1205):
        A = np.random.uniform(1, 5, (4, 4)) + np.eye(4) * 10
        # sau 1205 lần sinh ra 1205 ma trận khác nhau, nhưng chạy lại vẫn ra đúng 1205 ma trận đó
        r = analyze_matrix(A)
        assert r["Bankinhpho"] <= r["q"] + 1e-6


# 10 Đường chéo của B luôn bằng 0, chuanB = q
def test_cheo_B_bang_0_va_chuanB_bang_q():
    A = [[10, 2, 1], [3, 8, 2], [1, 1, 5]]
    # tính B như trên
    r = analyze_matrix(A)
    np.testing.assert_allclose(np.diag(r["matranB"]), 0)
    assert r["chuanB"] == pytest.approx(r["q"])


# 11 Đầu vào nguyên (int) vẫn chạy đúng
# hạn chế trường hợp phép chia lấy phần nguyên
# vd: 1/3 mà bị làm tròn thành 0 thay vì 0,333333333333 thì hệ số q sẽ sai
def test_dau_vao_so_nguyen():
    r = analyze_matrix([[3, 1], [1, 3]])
    assert r["q"] == pytest.approx(1/3)
    # kiểm tra đã đổi dữ liệu đầu vào sang số thực dtype=float chưa


# 12 Ma trận 1x1
def test_ma_tran_1x1():
    r = analyze_matrix([[12052006]])
    assert r["q"] == 0.0
    assert r["Bankinhpho"] == 0.0
    assert r["doixungg"]


# 13-16 Các trường hợp phải báo lỗi
def test_khong_vuong():
    with pytest.raises(ValueError):
        analyze_matrix([[1, 2, 3], [4, 5, 6]])


def test_khong_phai_ma_tran_2_chieu():
    with pytest.raises(ValueError):
        analyze_matrix([1, 2, 3])


def test_duong_cheo_bang_0():
    with pytest.raises(ValueError):
        analyze_matrix([[0, 1], [1, 2]])


def test_duong_cheo_rat_nho():
    with pytest.raises(ValueError):
        analyze_matrix([[1e-15, 1], [1, 2]])
test_functions = [
    test_vi_du_chuan, test_ma_tran_duong_cheo, test_doi_xung_2x2,
    test_khong_cheo_troi, test_cheo_troi_dau_bang, test_duong_cheo_am,
    test_gia_tri_rieng_phuc, test_lam_tron_6_chu_so, test_bkpho_nho_hon_hoac_bang_q,
    test_cheo_B_bang_0_va_chuanB_bang_q, test_dau_vao_so_nguyen,
    test_ma_tran_1x1, test_khong_vuong, test_khong_phai_ma_tran_2_chieu,
    test_duong_cheo_bang_0, test_duong_cheo_rat_nho
]

for func in test_functions:
    func()
    print(f"✔ {func.__name__} ---> PASSED")

print("pass")



✔ test_vi_du_chuan ---> PASSED
✔ test_ma_tran_duong_cheo ---> PASSED
✔ test_doi_xung_2x2 ---> PASSED
✔ test_khong_cheo_troi ---> PASSED
✔ test_cheo_troi_dau_bang ---> PASSED
✔ test_duong_cheo_am ---> PASSED
✔ test_gia_tri_rieng_phuc ---> PASSED
✔ test_lam_tron_6_chu_so ---> PASSED
✔ test_bkpho_nho_hon_hoac_bang_q ---> PASSED
✔ test_cheo_B_bang_0_va_chuanB_bang_q ---> PASSED
✔ test_dau_vao_so_nguyen ---> PASSED
✔ test_ma_tran_1x1 ---> PASSED
✔ test_khong_vuong ---> PASSED
✔ test_khong_phai_ma_tran_2_chieu ---> PASSED
✔ test_duong_cheo_bang_0 ---> PASSED
✔ test_duong_cheo_rat_nho ---> PASSED
pass
